# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/afreensumai64/ML-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [8]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN loaded successfully:", HF_TOKEN is not None)

HF_TOKEN loaded successfully: True





I use Logistic Regression because the task is to rank content observations by their likelihood of the observed March outcome proxy. Logistic Regression is fast, interpretable, and produces a probability score that can be used to create a ranked review queue.

The model uses observable February 2026 page-level signals and does not use future-window information, label-derived fields, or product-generated priority fields.

The model will be compared against the transparent Week-4 baseline using the same held-out evaluation population and Precision@50.

In [9]:
# W5 setup: connect to the FlyRank internship warehouse

import os
import duckdb
import pandas as pd
import numpy as np

from google.colab import userdata

# Get the Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# Create DuckDB connection
con = duckdb.connect()

# Authenticate with Hugging Face
con.execute(
    f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

# Warehouse location
REL = "hf://datasets/FlyRank/internship-warehouse"

# February 2026 = decision-time feature window
FEB = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-02/**/*.parquet'"
    f")"
)

# March 2026 = outcome window
MAR = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/**/*.parquet'"
    f")"
)

# Content dimension
DIM_CONTENT = (
    f"{REL}/dim_content/**/*.parquet"
)

print("DuckDB connection: READY")
print("February source: READY")
print("March source: READY")
print("Content dimension: READY")

DuckDB connection: READY
February source: READY
March source: READY
Content dimension: READY


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [12]:
# Check the actual files available in the FlyRank warehouse

print("Checking warehouse structure...")

files = con.sql(f"""
SELECT file
FROM glob('{REL}/**')
LIMIT 30
""").df()

display(files)

Checking warehouse structure...


,file
0,hf://datasets/FlyRank/internship-warehouse/.gi...
1,hf://datasets/FlyRank/internship-warehouse/REA...
2,hf://datasets/FlyRank/internship-warehouse/dim...
3,hf://datasets/FlyRank/internship-warehouse/dim...
4,hf://datasets/FlyRank/internship-warehouse/fac...
5,hf://datasets/FlyRank/internship-warehouse/fac...
6,hf://datasets/FlyRank/internship-warehouse/fac...
7,hf://datasets/FlyRank/internship-warehouse/fac...
8,hf://datasets/FlyRank/internship-warehouse/fac...
9,hf://datasets/FlyRank/internship-warehouse/fac...


In [13]:
# Find the exact dim_content path

files = con.sql(f"""
SELECT file
FROM glob('{REL}/**')
WHERE file ILIKE '%dim_content%'
""").df()

pd.set_option("display.max_colwidth", None)

display(files)

,file
0,hf://datasets/FlyRank/internship-warehouse/dim_content.parquet


In [14]:
# Build February decision-time features

DIM_CONTENT = f"{REL}/dim_content.parquet"

FEB_FEATURES = con.sql(f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks
    FROM {FEB}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),

content AS (
    SELECT
        client_hash_id,
        content_hash_id,
        content_created_date
    FROM read_parquet('{DIM_CONTENT}')
)

SELECT
    f.client_hash_id,
    f.content_hash_id,
    f.gsc_impressions,
    f.gsc_clicks,

    DATE_DIFF(
        'day',
        c.content_created_date,
        DATE '2026-02-28'
    ) AS content_age_days,

    CASE
        WHEN DATE_DIFF(
            'day',
            c.content_created_date,
            DATE '2026-02-28'
        ) >= 365
        AND f.gsc_impressions < 100
            THEN 4

        WHEN DATE_DIFF(
            'day',
            c.content_created_date,
            DATE '2026-02-28'
        ) >= 365
        AND f.gsc_impressions < 1000
            THEN 3

        WHEN DATE_DIFF(
            'day',
            c.content_created_date,
            DATE '2026-02-28'
        ) BETWEEN 180 AND 364
        AND f.gsc_impressions < 100
            THEN 3

        WHEN DATE_DIFF(
            'day',
            c.content_created_date,
            DATE '2026-02-28'
        ) BETWEEN 180 AND 364
        AND f.gsc_impressions < 1000
            THEN 2

        WHEN f.gsc_impressions < 100
            THEN 2

        WHEN f.gsc_impressions < 1000
            THEN 1

        ELSE 0
    END AS baseline_score

FROM feb f
JOIN content c
    ON f.client_hash_id = c.client_hash_id
   AND f.content_hash_id = c.content_hash_id

WHERE c.content_created_date IS NOT NULL
""").df()

print("February feature rows:", len(FEB_FEATURES))
display(FEB_FEATURES.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

February feature rows: 153559


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,content_age_days,baseline_score
0,client_e547b89c05043229,content_1eea820697c3b95a,299.0,0.0,226,2
1,client_e547b89c05043229,content_9abd8b303f805847,733.0,6.0,226,2
2,client_e547b89c05043229,content_5f58c55cbfee172a,514.0,0.0,226,2
3,client_e547b89c05043229,content_6fe390ba3af1e456,2931.0,3.0,226,0
4,client_e547b89c05043229,content_3ad5d2160242b9ca,970.0,2.0,226,2


In [15]:
# Build the March observed outcome proxy

MAR_OUTCOME = con.sql(f"""
WITH mar AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS march_clicks
    FROM {MAR}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    client_hash_id,
    content_hash_id,

    CASE
        WHEN COALESCE(march_clicks, 0) = 0 THEN 1
        ELSE 0
    END AS went_dark

FROM mar
""").df()

print("March outcome rows:", len(MAR_OUTCOME))
print("March went_dark rate:", MAR_OUTCOME["went_dark"].mean())

display(MAR_OUTCOME.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March outcome rows: 176738
March went_dark rate: 0.6105138679853795


,client_hash_id,content_hash_id,went_dark
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,1
1,client_62f4a7e64f5e0096,content_c03ecafd4c999f15,0
2,client_62f4a7e64f5e0096,content_e689bc511192751a,1
3,client_62f4a7e64f5e0096,content_7dbc094b799e05a4,0
4,client_62f4a7e64f5e0096,content_40b10da45f4c1cb5,1


In [16]:
# Combine February decision-time features with the March outcome

model_df = FEB_FEATURES.merge(
    MAR_OUTCOME,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print("Final modeling rows:", len(model_df))
print("Final went_dark rate:", model_df["went_dark"].mean())

display(model_df.head())

Final modeling rows: 134238
Final went_dark rate: 0.5683562031615489


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,content_age_days,baseline_score,went_dark
0,client_e547b89c05043229,content_1eea820697c3b95a,299.0,0.0,226,2,1
1,client_e547b89c05043229,content_9abd8b303f805847,733.0,6.0,226,2,0
2,client_e547b89c05043229,content_5f58c55cbfee172a,514.0,0.0,226,2,1
3,client_e547b89c05043229,content_6fe390ba3af1e456,2931.0,3.0,226,0,0
4,client_e547b89c05043229,content_3ad5d2160242b9ca,970.0,2.0,226,2,0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Train Logistic Regression and compare it with the Week-4 baseline

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
import numpy as np
import pandas as pd

# Features available at the February decision time
feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "content_age_days"
]

# Keep only rows with complete feature/outcome data
model_df = model_df.dropna(
    subset=feature_cols + ["went_dark", "client_hash_id"]
).copy()

X = model_df[feature_cols]
y = model_df["went_dark"]
groups = model_df["client_hash_id"]

# Grouped 80/20 split by client
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))
print("Train positive rate:", round(y_train.mean(), 4))
print("Test positive rate:", round(y_test.mean(), 4))


# Logistic Regression model
model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic_regression", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

model.fit(X_train, y_train)

# Model probability scores
model_scores = model.predict_proba(X_test)[:, 1]


# Precision@50 function
def precision_at_k(y_true, scores, k=50):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    order = np.argsort(scores)[::-1][:k]

    return y_true[order].mean()


# Model metrics
model_precision_50 = precision_at_k(
    y_test,
    model_scores,
    k=50
)

model_ap = average_precision_score(
    y_test,
    model_scores
)


# Same held-out test population for the baseline
test_results = model_df.iloc[test_idx].copy()

baseline_scores = test_results["baseline_score"].to_numpy()

baseline_precision_50 = precision_at_k(
    test_results["went_dark"],
    baseline_scores,
    k=50
)

baseline_ap = average_precision_score(
    test_results["went_dark"],
    baseline_scores
)


# Final comparison table
comparison = pd.DataFrame({
    "Method": [
        "Week-4 Baseline",
        "Logistic Regression"
    ],
    "Precision@50": [
        baseline_precision_50,
        model_precision_50
    ],
    "Average Precision": [
        baseline_ap,
        model_ap
    ]
})

display(comparison)


Train rows: 88344
Test rows: 45894
Train positive rate: 0.5402
Test positive rate: 0.6226


,Method,Precision@50,Average Precision
0,Week-4 Baseline,0.84,0.820167
1,Logistic Regression,1.00,0.897893


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Error analysis on the held-out test set

test_results["model_score"] = model_scores

test_results["predicted_class"] = (
    test_results["model_score"] >= 0.5
).astype(int)

false_positives = test_results[
    (test_results["predicted_class"] == 1) &
    (test_results["went_dark"] == 0)
]

false_negatives = test_results[
    (test_results["predicted_class"] == 0) &
    (test_results["went_dark"] == 1)
]

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print("\nTop false positives:")
display(
    false_positives.sort_values(
        "model_score",
        ascending=False
    )[
        [
            "client_hash_id",
            "content_hash_id",
            "model_score",
            "went_dark"
        ]
    ].head(10)
)

print("\nTop false negatives:")
display(
    false_negatives.sort_values(
        "model_score",
        ascending=False
    )[
        [
            "client_hash_id",
            "content_hash_id",
            "model_score",
            "went_dark"
        ]
    ].head(10)
)


False positives: 6637
False negatives: 1364

Top false positives:


,client_hash_id,content_hash_id,model_score,went_dark
77814,client_9958f0a7ae1df715,content_063b75a468ef5008,0.823260,0
38815,client_9958f0a7ae1df715,content_7990908859880eea,0.821941,0
64865,client_3ffa76342f366962,content_58dfc8ab3f7b8efe,0.821060,0
38645,client_3ffa76342f366962,content_32d2e8e66805c5fc,0.821060,0
110242,client_3ffa76342f366962,content_28ad4b3562dde1f2,0.821060,0
12965,client_3ffa76342f366962,content_1ddf0f46fc5665f1,0.821060,0
79889,client_3ffa76342f366962,content_dae098df4ba4112d,0.821060,0
110248,client_3ffa76342f366962,content_55331add435a3468,0.821060,0
131711,client_3ffa76342f366962,content_9c210cac37751270,0.821060,0
105740,client_3ffa76342f366962,content_16c279df763c4803,0.821060,0



Top false negatives:


,client_hash_id,content_hash_id,model_score,went_dark
39331,client_73cda7b4e4f265ea,content_c21662ad65d2d190,0.499997,1
82821,client_73cda7b4e4f265ea,content_02ee5d0cc5f90db4,0.499806,1
87292,client_73cda7b4e4f265ea,content_9416e6131123df98,0.499723,1
20224,client_73cda7b4e4f265ea,content_36c3dcefd24b1774,0.499441,1
75272,client_b10cb2997d0c7c86,content_51eb42f079ad919a,0.499270,1
13549,client_73cda7b4e4f265ea,content_cde30c28a96d7486,0.499043,1
8565,client_3ffa76342f366962,content_d27e01f88523ee5b,0.498880,1
11656,client_73cda7b4e4f265ea,content_280851b6fed47799,0.498746,1
85949,client_73cda7b4e4f265ea,content_1f66e1a4f6d5b819,0.498314,1
78895,client_73cda7b4e4f265ea,content_9dac4d3f79d355ce,0.497906,1


### Error interpretation

The model produced 6,637 false positives and 1,364 false negatives on the held-out test set.

Several high-scored false positives did not show the observed March outcome. This indicates that the available February signals can identify pages with similar observed characteristics, but those characteristics do not guarantee the subsequent outcome.

The false negatives shown above have predicted probabilities close to the 0.50 classification threshold, indicating that some observations are borderline cases under this threshold.

These errors reinforce that the model should be used as a prioritization and review aid rather than as an automatic content-refresh decision.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card.